In [1]:
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.api import VAR
import pandas as pd
import numpy as np

In [5]:
df = pd.read_csv("../../data/FRB_H15.csv").dropna()

#rename columns
df.rename(columns={"Series Description": "Date", "Market yield on U.S. Treasury securities at 1-month  constant maturity, quoted on investment basis": "0Y1M", "Market yield on U.S. Treasury securities at 3-month  constant maturity, quoted on investment basis": "0Y3M", "Market yield on U.S. Treasury securities at 6-month  constant maturity, quoted on investment basis": "0Y6M", "Market yield on U.S. Treasury securities at 1-year  constant maturity, quoted on investment basis": "1Y", "Market yield on U.S. Treasury securities at 2-year  constant maturity, quoted on investment basis": "2Y", "Market yield on U.S. Treasury securities at 3-year  constant maturity, quoted on investment basis": "3Y", "Market yield on U.S. Treasury securities at 5-year  constant maturity, quoted on investment basis": "5Y", "Market yield on U.S. Treasury securities at 7-year  constant maturity, quoted on investment basis": "7Y", "Market yield on U.S. Treasury securities at 10-year  constant maturity, quoted on investment basis": "10Y", "Market yield on U.S. Treasury securities at 20-year constant maturity, quoted on investment basis": "20Y", "Market yield on U.S. Treasury securities at 30-year  constant maturity, quoted on investment basis": "30Y"}, inplace=True)
df.rename(columns={"Market yield on U.S. Treasury securities at 20-year  constant maturity, quoted on investment basis": "20Y"}, inplace=True)

#make index to datetime for timeseries
df["Date"] = pd.to_datetime(df["Date"])
df.set_index("Date", inplace=True)

df.tail(5)

,0Y1M,0Y3M,0Y6M,1Y,2Y,3Y,5Y,7Y,10Y,20Y,30Y
Date,,,,,,,,,,,
2026-05-08,3.71,3.69,3.74,3.75,3.90,3.92,4.02,4.19,4.38,4.93,4.95
2026-05-11,3.71,3.70,3.77,3.79,3.95,3.96,4.07,4.24,4.42,4.97,4.98
2026-05-12,3.71,3.70,3.77,3.80,4.00,4.01,4.12,4.29,4.46,5.02,5.03
2026-05-13,3.71,3.69,3.77,3.79,3.98,4.00,4.12,4.28,4.46,5.03,5.03
2026-05-14,3.72,3.69,3.76,3.79,4.00,4.04,4.13,4.29,4.47,5.01,5.02


In [14]:
#split data int training and testing sets

trainSplit = {}
#maturities_list = []
for col in df.columns:
    #maturities[col] = df[[col]]
    #maturities_list.append(df[[col]])

    curr = df[[col]]
    trainSplit[col] = {"train": curr[curr.index <= "2025-4-17"], "test": curr[curr.index > "2025-4-17"]} 
    
    for i in range(20):
        new_row = pd.DataFrame({col: [trainSplit[col]["train"][col].iloc[-1]]})
        #add in 20 new rows based on the last given data (for a random walk baseline)
        trainSplit[col]["train"] = pd.concat([trainSplit[col]["train"], new_row])

matList = ["0Y1M", "0Y3M", "0Y6M", "1Y", "2Y", "3Y", "5Y", "7Y", "10Y", "20Y", "30Y"]

## Random Walk model

In [15]:
#creating a random walk time series for baseline
#assume previous day is the same value as today 

#last 20 values of trainsplit in the df is the random walk model (next 20 days)

testArr = []
for mat in matList:
    testArr.append(trainSplit[mat]["test"][mat].to_numpy())

horizons = []
errors = []
rmse = []

for i in range(len(matList)):

    horizons.append([])
    errors.append([])
    rmse.append([])

    #the random walk forecasted number for day 1,5,20
    horizons[i].extend(((trainSplit[matList[i]]["train"][matList[i]].iloc[-20]), (trainSplit[matList[i]]["train"][matList[i]].iloc[-16]), (trainSplit[matList[i]]["train"][matList[i]].iloc[-1])))
    errors[i].extend(((horizons[i][0] - testArr[i][0]), (horizons[i][1] - testArr[i][4]), (horizons[i][2] - testArr[i][19])))
    rmse[i].extend(((np.sqrt(np.mean(errors[i][0]**2))), (np.sqrt(np.mean(errors[i][1]**2))), (np.sqrt(np.mean(errors[i][2]**2)))))

    print("Maturity: ", matList[i], "\n RMSE for 1 day forecast: ", rmse[i][0], "\n RMSE for 5 day forecast: ", rmse[i][1], "\n RMSE for 20 day forecast: ", rmse[i][2], "\n\n")

#rmse = np.sqrt((errors**2).mean())
#mae = errors.abs().mean()
#print("MAE for random walk forecast(baseline): \n", mae)
#print("RMSE for random walk ", rmse)

Maturity:  0Y1M 
 RMSE for 1 day forecast:  0.010000000000000675 
 RMSE for 5 day forecast:  0.020000000000000462 
 RMSE for 20 day forecast:  0.009999999999999787 


Maturity:  0Y3M 
 RMSE for 1 day forecast:  0.0 
 RMSE for 5 day forecast:  0.019999999999999574 
 RMSE for 20 day forecast:  0.03000000000000025 


Maturity:  0Y6M 
 RMSE for 1 day forecast:  0.009999999999999787 
 RMSE for 5 day forecast:  0.0 
 RMSE for 20 day forecast:  0.08000000000000007 


Maturity:  1Y 
 RMSE for 1 day forecast:  0.040000000000000036 
 RMSE for 5 day forecast:  0.040000000000000036 
 RMSE for 20 day forecast:  0.13999999999999968 


Maturity:  2Y 
 RMSE for 1 day forecast:  0.06000000000000005 
 RMSE for 5 day forecast:  0.06999999999999984 
 RMSE for 20 day forecast:  0.16999999999999993 


Maturity:  3Y 
 RMSE for 1 day forecast:  0.04999999999999982 
 RMSE for 5 day forecast:  0.06000000000000005 
 RMSE for 20 day forecast:  0.13000000000000034 


Maturity:  5Y 
 RMSE for 1 day forecast:  0.020